In [1]:
from transformers import BartTokenizer
from datasets import Dataset
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from scripts.Utils import TimexNorm_Utils
from scripts.Reader import obtain_combined_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

In [2]:
tokenizer.add_special_tokens({
  "additional_special_tokens": ["<timex","type=DATE>","type=TIME>",
                                "type=DURATION>","type=SET>","</timex>", "<sep>"]
})
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


BartScaledWordEmbedding(50272, 768, padding_idx=1)

In [3]:
datasets = obtain_combined_dataset(["TempEval3","wikiwars","tweets"], "normalised")

In [5]:
datasets

DatasetDict({
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 739
    })
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 15843
    })
    eval: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 1822
    })
})

In [6]:
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/739 [00:00<?, ? examples/s]

Map:   0%|          | 0/15843 [00:00<?, ? examples/s]

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=1,
    predict_with_generate=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    compute_metrics=utils.compute_metrics,
)

In [10]:
trainer.train()

Step,Training Loss
500,0.211000


d:\GeoTKG\venv\Lib\site-packages\transformers\modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=991, training_loss=0.16463031249137747, metrics={'train_runtime': 227.2352, 'train_samples_per_second': 69.721, 'train_steps_per_second': 4.361, 'total_flos': 4824689778155520.0, 'train_loss': 0.16463031249137747, 'epoch': 1.0})

In [11]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

('./results/TimeNormBart\\tokenizer_config.json',
 './results/TimeNormBart\\special_tokens_map.json',
 './results/TimeNormBart\\vocab.json',
 './results/TimeNormBart\\merges.txt',
 './results/TimeNormBart\\added_tokens.json')

In [12]:
predictions_output = trainer.predict(datasets["test"])
# Returns a namedtuple with:
#  - predictions_output.predictions: raw token‑ID logits or IDs (depending on config)
#  - predictions_output.label_ids: the gold token IDs

# 4) Decode predictions into strings
decoded_preds = tokenizer.batch_decode(
    predictions_output.predictions, 
    skip_special_tokens=True
)
decoded_labels = tokenizer.batch_decode(
    predictions_output.label_ids, 
    skip_special_tokens=True
)

for p, t in zip(decoded_preds, decoded_labels):
    print(f"P: {p} | T: {t}")

# 5) Compute a simple exact‑match accuracy
exact_match = sum(p == t for p, t in zip(decoded_preds, decoded_labels)) / len(decoded_labels)
print(f"Exact‑match accuracy: {exact_match:.2%}")

d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 1969-06 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 1920-09-21 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 2015-03-09 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 1935-10-11 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 1979-07 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
d:\GeoTKG\venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 201

P: 1969-06 | T: 1969-06
P: 1944-09-21 | T: 1920-09-21
P: 2015-03-09 | T: 2015-03-09
P: 1918-10-11 | T: 1935-10-11
P: 1980-07 | T: 1979-07
P: 2014-09-30 | T: 2014-09-30
P: PT1hr | T: P1H
P: PRESENT_REF | T: PAST_REF
P: P1D | T: P1D
P: PXM | T: PXM
P: P2W | T: P1W
P: 2014-12-21 | T: 2014-12-25
P: 2012-W47-WE | T: 2012-W13-WE
P: 1985-04 | T: 1985-04
P: 1944-08-17 | T: 1920-08-17
P: 1920-08-06 | T: 1920-08-06
P: 2009-12-19 | T: XXXX-XX-XX
P: 1918-11-10 | T: 1920-11-10
P: 1975-10-12 | T: 1967-10-12
P: PRESENT_REF | T: PRESENT_REF
P: 1918-08-01 | T: 1920-08-01
P: 2013-04-07 | T: 2013-04-07
P: 1904-04-13 | T: 1904-04-13
P: 1935-10-03T05:00 | T: 1935-10-03T05:00
P: 1944-04 | T: 1919-04
P: PRESENT_REF | T: PRESENT_REF
P: 1989-02-15 | T: 1989-02-15
P: 2013-11-01 | T: 2013-11-1
P: 2013-03-22TAF | T: 2013-03-22TAF
P: 2013-05 | T: 2012-05
P: PT24H | T: PT24H
P: 1936-03-31 | T: 1936-03-31
P: 2014-02-21 | T: 2014-02-21
P: 2015-XX-XX | T: 2015
P: 1944-03 | T: 1920-03
P: PRESENT_REF | T: PRESENT_REF
P:

In [13]:
from seqeval.metrics import f1_score, precision_score, recall_score

# 4) Flatten to single lists
list_preds  = [[sub] for sub in decoded_preds]
list_labels = [[sub] for sub in decoded_labels]

print(f1_score(list_labels, list_preds, mode="strict"))

0.7713125845737483


In [ ]:
list_labels